In [13]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [14]:
model_path = "../simple_with_activation_linear_model.onnx"

In [15]:
# Create a simple one layer model using a linear layer


class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        # Define a single linear layer
        self.linear = nn.Linear(input_size, 10)
        self.act_1_relu = nn.ReLU()
        self.linear2 = nn.Linear(10, 20)
        self.act_2_sigmoid = nn.Sigmoid()
        self.linear3 = nn.Linear(20, 15)
        self.act_3_tanh = nn.Tanh()
        self.linear4 = nn.Linear(15, output_size)
        self.act_4_softmax = nn.Softmax(dim=1)  # Apply softmax along the feature dimension

    def forward(self, x):
        # Pass input through the linear layer
        output = self.linear(x)
        output = self.act_1_relu(output)
        output = self.linear2(output)
        output = self.act_2_sigmoid(output)
        output = self.linear3(output)
        output = self.act_3_tanh(output)
        output = self.linear4(output)
        output = self.act_4_softmax(output)
        return output


# Example usage
model = SimpleLinearModel(input_size=10, output_size=5)
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleLinearModel(
  (linear): Linear(in_features=10, out_features=10, bias=True)
  (act_1_relu): ReLU()
  (linear2): Linear(in_features=10, out_features=20, bias=True)
  (act_2_sigmoid): Sigmoid()
  (linear3): Linear(in_features=20, out_features=15, bias=True)
  (act_3_tanh): Tanh()
  (linear4): Linear(in_features=15, out_features=5, bias=True)
  (act_4_softmax): Softmax(dim=1)
)
Model weights:
linear.weight: torch.Size([10, 10])
  Weight values (first 5): tensor([ 0.1052,  0.2601,  0.0438,  0.1561, -0.3066], grad_fn=<SliceBackward0>)
linear.bias: torch.Size([10])
  Bias values (first 5): tensor([ 0.1445, -0.2364,  0.2843,  0.1374, -0.0510], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([20, 10])
  Weight values (first 5): tensor([-0.0496,  0.1413,  0.2947,  0.1072,  0.1886], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([20])
  Bias values (first 5): tensor([-0.1536, -0.0011,  0.2609, -0.0672,  0.1879], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([15, 20])
  Weig

In [16]:
# export to onnx
onnx.export(model, torch.randn(1, 10), model_path, export_params=True, opset_version=11)

In [17]:
# run the model with pytorch
input_data = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]])
with torch.no_grad():
    output = model(input_data)
print(output)

tensor([[0.1537, 0.2518, 0.2072, 0.2191, 0.1682]])


In [18]:
# from c_exporter.onnx_exporter import export_onnx

# output_model = export_onnx(model_path)
# print(output_model)